# Redrob Ranker — sandbox demo

Runs the full ranking pipeline end-to-end on a small candidate sample (≤100 candidates), on CPU, with no network access during ranking, and produces a ranked `submission.csv`.

**How to use:** Runtime → Run all. Takes well under a minute. To try your own sample, run the upload cell and pick any `candidates` file (`.json` array or `.jsonl`).

In [ ]:
# 1. Get the code
!git clone --depth 1 https://github.com/k25kar/redrob-ranker.git
%cd redrob-ranker
!pip -q install -r requirements.txt

In [ ]:
# 2. Candidate sample.
# Default: the 50-candidate sample bundled in this repo's sandbox/ folder.
# To use your own file instead, set UPLOAD = True and run this cell.
UPLOAD = False
src = 'sandbox/sample_candidates.json'
if UPLOAD:
    from google.colab import files
    up = files.upload()
    src = list(up.keys())[0]
!python scripts/make_sample.py --input "{src}" --n 100 --out data/sample.jsonl

In [ ]:
# 3. Rank (the actual ranking step: CPU-only, no network, deterministic)
!python scripts/rank.py --candidates data/sample.jsonl

In [ ]:
# 4. Inspect the output
import pandas as pd
sub = pd.read_csv('outputs/submission.csv')
print(f'{len(sub)} rows; score monotone: {(sub.score.diff().dropna() <= 0).all()}')
sub.head(15)

In [ ]:
# 5. Determinism check: run again, byte-compare
import hashlib, shutil
h1 = hashlib.sha256(open('outputs/submission.csv','rb').read()).hexdigest()
!python scripts/rank.py --candidates data/sample.jsonl > /dev/null
h2 = hashlib.sha256(open('outputs/submission.csv','rb').read()).hexdigest()
print('identical across runs:', h1 == h2)